# ByteRCNN — FFT-75 Scenario #1 Baseline

**Frozen benchmark**: 512-byte fragments · 75 classes · 80/10/10 split · seed=42

This notebook:
1. Installs dependencies
2. Downloads the FFT-75 dataset from Google Drive with `gdown`
3. Unpacks and validates the raw fragments
4. Builds the frozen deterministic split
5. Runs a sanity training pass (2 epochs, 2 000 samples)
6. Runs full training (30 epochs, early stopping)
7. Evaluates on the frozen test set
8. Saves all outputs

> **Before running**: replace `GDRIVE_FILE_ID` in Cell 3 with your actual Google Drive file or folder ID.

## Cell 1 — Install Dependencies

In [ ]:
# ── Install / upgrade required packages ──────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('gdown>=5.2.0')
pip('pyyaml>=6.0')
pip('scikit-learn>=1.5')
pip('pandas>=2.2')
pip('numpy>=1.26')
pip('matplotlib>=3.9')
pip('tqdm>=4.66')
pip('seaborn>=0.13')

# PyTorch is pre-installed on Kaggle GPU images — verify:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

## Cell 2 — Clone / Mount the DeepCarv Repo

In [ ]:
# ── Clone repo (or use Kaggle dataset mount if you've added it as a dataset) ──
import os
from pathlib import Path

WORKING = Path('/kaggle/working')
REPO_DIR = WORKING / 'deepcarv'

if not REPO_DIR.exists():
    # Replace with your actual repo URL if it's public, or
    # mount the repo as a Kaggle dataset and set REPO_DIR accordingly.
    !git clone --depth 1 https://github.com/YOUR_USERNAME/deepcarv.git {REPO_DIR}
else:
    print('Repo already present:', REPO_DIR)

# Add repo root to Python path
import sys
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('sys.path[0]:', sys.path[0])

# Tell DeepCarv path resolver we are in Kaggle runtime
os.environ['KAGGLE_RUNTIME'] = '1'

## Cell 3 — Configure Paths & Google Drive ID

**Replace `YOUR_GDRIVE_FILE_ID` with your actual Google Drive file or folder ID.**

- For a **single zip file** shared via Drive: use the file ID from the share URL.
- For a **Drive folder**: use the folder ID and set `IS_FOLDER = True`.

In [ ]:
# ── !! CONFIGURE HERE !! ────────────────────────────────────────────────────
GDRIVE_FILE_ID  = 'YOUR_GDRIVE_FILE_ID'   # <-- replace this
ARCHIVE_NAME    = 'FFT-75.zip'             # expected filename after download
IS_FOLDER       = False                    # True if GDRIVE_FILE_ID is a folder
# ────────────────────────────────────────────────────────────────────────────

RAW_DIR      = Path('/kaggle/working/data/raw')
SPLITS_DIR   = Path('/kaggle/working/data/splits')
CKPT_DIR     = Path('/kaggle/working/checkpoints')
OUTPUTS_DIR  = Path('/kaggle/working/outputs')
LOGS_DIR     = Path('/kaggle/working/logs')

for d in [RAW_DIR, SPLITS_DIR, CKPT_DIR, OUTPUTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Paths configured:')
for name, p in [('RAW_DIR', RAW_DIR), ('SPLITS_DIR', SPLITS_DIR),
                ('CKPT_DIR', CKPT_DIR), ('OUTPUTS_DIR', OUTPUTS_DIR)]:
    print(f'  {name:<15} {p}')

## Cell 4 — Download Dataset from Google Drive

In [ ]:
import gdown

archive_path = WORKING / 'data' / ARCHIVE_NAME

if archive_path.exists():
    print(f'Archive already downloaded: {archive_path}')
else:
    print(f'Downloading {ARCHIVE_NAME} from Google Drive …')
    if IS_FOLDER:
        gdown.download_folder(
            id=GDRIVE_FILE_ID,
            output=str(RAW_DIR),
            quiet=False,
        )
        print('Folder download complete.')
    else:
        gdown.download(
            id=GDRIVE_FILE_ID,
            output=str(archive_path),
            quiet=False,
        )
        print(f'Downloaded → {archive_path}  ({archive_path.stat().st_size / 1e6:.1f} MB)')

## Cell 5 — Unpack and Validate the Dataset

In [ ]:
import zipfile, tarfile, shutil

archive_path = WORKING / 'data' / ARCHIVE_NAME

def unpack_archive(src: Path, dst: Path):
    dst.mkdir(parents=True, exist_ok=True)
    name = src.name.lower()
    if name.endswith('.zip'):
        print(f'Extracting ZIP → {dst}')
        with zipfile.ZipFile(src) as z:
            z.extractall(dst)
    elif name.endswith(('.tar.gz', '.tgz', '.tar.bz2', '.tar')):
        print(f'Extracting TAR → {dst}')
        with tarfile.open(src) as t:
            t.extractall(dst)
    else:
        print(f'Unrecognised archive format: {src.name}')
        print('If the dataset was downloaded as a folder, skip this cell.')

if archive_path.exists():
    unpack_archive(archive_path, RAW_DIR)
else:
    print('No archive found — assuming folder download is already in RAW_DIR.')

# ── Validation: count 512-byte files ────────────────────────────────────────
fragment_files = [
    f for f in RAW_DIR.rglob('*')
    if f.is_file() and f.stat().st_size == 512
]
print(f'\nValidation: found {len(fragment_files):,} files of exactly 512 bytes under {RAW_DIR}')

# Count unique subdirs (= classes)
subdirs = set(f.parent.name for f in fragment_files if f.parent != RAW_DIR)
if subdirs:
    print(f'Unique class folders : {len(subdirs)}')
    print('Sample classes       :', sorted(subdirs)[:10])

if len(fragment_files) == 0:
    raise RuntimeError(
        'No 512-byte fragment files found!\n'
        'Check the Drive ID and archive structure.'
    )

## Cell 6 — Build Frozen Benchmark Split

Builds `train.csv`, `val.csv`, `test.csv`, `class_map.json`, `manifest.json` 
with the **frozen** seed=42, 80/10/10 split, Scenario #1.

In [ ]:
from src.data.build_fft75_split import build_split

build_split(
    raw_dir   = RAW_DIR,
    splits_dir = SPLITS_DIR,
    seed      = 42,      # FROZEN — do not change
    force     = True,    # overwrite if re-running this cell
)

# Quick verification
import pandas as pd
fft75_dir = SPLITS_DIR / 'fft75_s1_512'
for split in ('train', 'val', 'test'):
    df = pd.read_csv(fft75_dir / f'{split}.csv')
    print(f'  {split:5s}  {len(df):>8,} rows   classes={df["label_id"].nunique()}')

## Cell 7 — Sanity Training Run

2 epochs · 2 000 samples · verifies end-to-end pipeline health.

In [ ]:
import os
os.environ['KAGGLE_RUNTIME'] = '1'

from src.training.sanity_train_bytercnn import main as sanity_main

sanity_main([
    '--train_csv',  str(fft75_dir / 'train.csv'),
    '--class_map',  str(fft75_dir / 'class_map.json'),
    '--checkpoint_path', str(CKPT_DIR / 'sanity_bytercnn_fft75.pt'),
    '--epochs', '2',
    '--subset', '2000',
    '--batch_size', '64',
])

## Cell 8 — Full Training (30 epochs, early stopping)

In [ ]:
from src.training.train_bytercnn import main as train_main

train_main([
    '--train_csv',       str(fft75_dir / 'train.csv'),
    '--val_csv',         str(fft75_dir / 'val.csv'),
    '--class_map',       str(fft75_dir / 'class_map.json'),
    '--checkpoint_path', str(CKPT_DIR / 'best_bytercnn_fft75.pt'),
    '--epochs',    '30',
    '--batch_size','256',
    '--lr',        '1e-3',
    '--patience',  '5',
    '--grad_clip', '1.0',
    '--seed',      '42',
])

## Cell 9 — Evaluation on Frozen Test Set

In [ ]:
from src.evaluation.evaluate_bytercnn import main as eval_main

eval_out = OUTPUTS_DIR / 'bytercnn_fft75'

eval_main([
    '--checkpoint', str(CKPT_DIR / 'best_bytercnn_fft75.pt'),
    '--test_csv',   str(fft75_dir / 'test.csv'),
    '--class_map',  str(fft75_dir / 'class_map.json'),
    '--out_dir',    str(eval_out),
    '--batch_size', '256',
    '--seed',       '42',
])

## Cell 10 — Display Results

In [ ]:
import json
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ── Scalar metrics ───────────────────────────────────────────────────────────
metrics_path = eval_out / 'metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print('=== Evaluation Metrics ===')
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f'  {k:<35} {v:.6f}')
        else:
            print(f'  {k:<35} {v}')

# ── Training curves ──────────────────────────────────────────────────────────
curves_path = OUTPUTS_DIR / 'bytercnn_fft75' / 'training_curves.png'
if curves_path.exists():
    img = mpimg.imread(str(curves_path))
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Curves')
    plt.tight_layout()
    plt.show()

# ── Per-class metrics (top 10 by F1) ─────────────────────────────────────────
per_class_path = eval_out / 'per_class_metrics.csv'
if per_class_path.exists():
    pc_df = pd.read_csv(per_class_path)
    print('\n=== Top-10 Classes by F1 ===')
    display(pc_df.sort_values('f1', ascending=False).head(10).reset_index(drop=True))
    print('\n=== Bottom-10 Classes by F1 ===')
    display(pc_df.sort_values('f1', ascending=True).head(10).reset_index(drop=True))

## Cell 11 — Package Outputs

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
bundle_name = f'bytercnn_fft75_results_{timestamp}.zip'
bundle_path = WORKING / bundle_name

# Collect outputs to bundle
bundle_dir = WORKING / f'bundle_{timestamp}'
bundle_dir.mkdir(exist_ok=True)

for fname in [
    'metrics.json',
    'confusion_matrix.csv',
    'per_class_metrics.csv',
    'predictions.csv',
    'eval_summary.txt',
]:
    src = eval_out / fname
    if src.exists():
        shutil.copy2(src, bundle_dir / fname)

curves = OUTPUTS_DIR / 'bytercnn_fft75' / 'training_curves.png'
if curves.exists():
    shutil.copy2(curves, bundle_dir / 'training_curves.png')

# Best checkpoint
best_ckpt = CKPT_DIR / 'best_bytercnn_fft75.pt'
if best_ckpt.exists():
    shutil.copy2(best_ckpt, bundle_dir / 'best_bytercnn_fft75.pt')

shutil.make_archive(str(bundle_path.with_suffix('')), 'zip', bundle_dir)
print(f'Results bundled → {bundle_path}')
print(f'Size: {bundle_path.stat().st_size / 1e6:.1f} MB')
print('\nFiles in bundle:')
for f in sorted(bundle_dir.iterdir()):
    print(f'  {f.name}')